In [1]:
import pandas_ta as pta
import numpy as np
import pandas as pd
import mplfinance as mpf
from tabulate import tabulate
from utils.KrakenHistoricalData import KrakenHistoricalData
from ggTrader.Signals import Signals
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
# Python
top20_kraken = [
    "BTC",   # 1 Bitcoin
    "ETH",   # 2 Ethereum
    "XRP",   # 3 XRP
    "BNB",   # 4 BNB
    "SOL",   # 5 Solana
    "TRX",   # 6 TRON
    "DOGE",  # 7 Dogecoin
    "ADA",   # 8 Cardano
    # "HYPE",  # 9 Hyperliquid
    "LINK",  # 10 Chainlink
    "BCH",   # 11 Bitcoin Cash
    "XLM",   # 12 Stellar
    "SUI",   # 13 Sui
    "HBAR",  # 14 Hedera
    "AVAX",  # 15 Avalanche
    "ZEC",   # 16 Zcash
    "LTC",   # 17 Litecoin
    "XMR",   # 18 Monero
    "SHIB",  # 19 Shiba Inu
    "TON",   # 20 Toncoin
    "CRO",   # 21 Crypto.com
    "DOT",   # 22 Polkadot
    "MNT",   # 23 Mantle
    "TAO",   # 24 Bittensor
    "UNI",   # 25 Uniswap
]

In [3]:
# Ticker
symbols = ["BTC", "ETH","DOT"]
symbols = top20_kraken
interval = "4h"

# Time Range

end = pd.to_datetime("2025-06-30").tz_localize('UTC')
start = end - pd.Timedelta(days=30 * 6)

k = KrakenHistoricalData()

# df_multi = k.get_ohlcv_df(symbols, interval=interval)
k.use_remote("https://garygigabytes.com/kraken/parquet")  # no trailing slash
df_multi = k.get_ohlcv_df_remote(symbols, interval=interval)



print(close)
print(f"\nMultiIndex")
print(df_multi.info())

# select all close
print(df_multi.xs('close', axis=1, level=1).head())

# select only BTC
print(df_multi.xs('BTC', axis=1, level=0).head())

# list of tickers
print(df_multi.columns.levels[0].tolist())

None

MultiIndex
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 6024 entries, 2023-01-01 00:00:00+00:00 to 2025-09-30 20:00:00+00:00
Freq: 4h
Columns: 192 entries, ('BTC', 'open') to ('UNI', 'quote')
dtypes: Int64(24), float32(96), float64(24), object(48)
memory usage: 6.9+ MB
None
                                    BTC          ETH      XRP  BNB   SOL  \
2023-01-01 00:00:00+00:00  16519.300781  1194.140015  0.33838  NaN  9.99   
2023-01-01 04:00:00+00:00  16510.699219  1192.119995  0.33586  NaN  9.82   
2023-01-01 08:00:00+00:00  16500.199219  1193.790039  0.33634  NaN  9.72   
2023-01-01 12:00:00+00:00  16549.900391  1197.270020  0.33821  NaN  9.95   
2023-01-01 16:00:00+00:00  16553.699219  1197.140015  0.33690  NaN  9.87   

                                TRX      DOGE       ADA     LINK        BCH  \
2023-01-01 00:00:00+00:00  0.054521  0.069783  0.244274  5.53086  96.480003   
2023-01-01 04:00:00+00:00  0.054577  0.069055  0.244368  5.50612  96.180000   
2023-01-01 08:00:

In [4]:
def process_ohlcv(df):
    data = {}
    col = ['open', 'high', 'low', 'close', 'volume']
    for c in col:
        data[c] = df.xs(c, axis=1, level=1)
    return data


data = process_ohlcv(df_multi)

print("\nProcessed Data")
for col in data.keys():
    print(f"{col}:")
    print(data[col].head())



Processed Data
open:
                                    BTC          ETH      XRP  BNB   SOL  \
2023-01-01 00:00:00+00:00  16528.699219  1195.000000  0.33867  NaN  9.97   
2023-01-01 04:00:00+00:00  16519.300781  1194.150024  0.33845  NaN  9.99   
2023-01-01 08:00:00+00:00  16512.400391  1192.199951  0.33586  NaN  9.82   
2023-01-01 12:00:00+00:00  16500.199219  1193.430054  0.33664  NaN  9.73   
2023-01-01 16:00:00+00:00  16550.000000  1197.280029  0.33825  NaN  9.94   

                                TRX      DOGE       ADA     LINK        BCH  \
2023-01-01 00:00:00+00:00  0.054435  0.070156  0.245368  5.56267  96.970001   
2023-01-01 04:00:00+00:00  0.054483  0.069580  0.244274  5.52101  96.400002   
2023-01-01 08:00:00+00:00  0.054595  0.069119  0.244342  5.50854  96.220001   
2023-01-01 12:00:00+00:00  0.054416  0.069510  0.244400  5.50978  96.440002   
2023-01-01 16:00:00+00:00  0.054721  0.069588  0.246023  5.56977  96.690002   

                           ...        ZEC     

In [5]:
# testing out how I can apply signals to the entire multiindex dataframe

def add_ticker_to_columns(ticker: str, df: pd.DataFrame) -> pd.DataFrame:
    df_cols = df.columns.tolist()
    new_cols = []
    for col in df_cols:
        new_cols.append((ticker, col.lower()))
    df.columns = pd.MultiIndex.from_tuples(new_cols)
    return df


def entry_signals(df: pd.DataFrame, chandelier_exit: pd.Series, adx_length: int = 14,
                  adx_threshold: int = 25) -> pd.DataFrame:
    signals = pd.DataFrame(index=df.index)

    # ADX
    adx = pta.adx(df['high'],
                  df['low'],
                  df['close'],
                  length=adx_length)
    signals['adx'] = adx.iloc[:, 0]
    signals['adx_d'] = adx.iloc[:, 1]
    signals['adx_dmp'] = adx.iloc[:, 2]
    signals['adx_dmn'] = adx.iloc[:, 3]
    signals['adx_signal'] = np.where(signals['adx'] > adx_threshold, True, False)
    signals['adx_bull'] = np.where(signals['adx_dmp'] > signals['adx_dmn'], True, False)
    signals['adx_signal'] = signals['adx_signal'] & signals['adx_bull']

    # PSAR
    psar = pta.psar(df['high'],
                    df['low'],
                    close=df['close'])
    signals['sar'] = psar.iloc[:, 0]
    signals['sar_signal'] = df['close'] > signals['sar']

    # entry

    # Have ADX strength, and sar is a buy and ce is not an exit
    entry_series = signals['adx_signal'] & signals['sar_signal'] & ~chandelier_exit
    # entry_series = signals['adx_signal'] & signals['sar_signal']
    signals['entry_rise'] = entry_series & (~entry_series.shift(1, fill_value=False))
    signals['entry_series'] = entry_series

    return signals


def exit_signals(df: pd.DataFrame, atr_multiplier: float = 3.0, ce_high_length: int = 22):
    signals = pd.DataFrame(index=df.index)

    # Exit: Chandelier Exit uses ATR
    ce = pta.chandelier_exit(df['high'],
                             df['low'],
                             df['close'],
                             multiplier=atr_multiplier, high_length=ce_high_length)

    signals['ce_l'] = np.where(ce.iloc[:, 2] > 0, ce.iloc[:, 0], np.nan)
    signals['ce_sh'] = np.where(ce.iloc[:, 2] < 0, ce.iloc[:, 1], np.nan)
    signals['ce_exit'] = np.where(ce.iloc[:, 2] == 1, False, True)

    exit_series = signals['ce_exit']

    signals['exit_rise'] = exit_series & (~exit_series.shift(1, fill_value=False))
    signals['exit_series'] = exit_series
    return signals


def filter_signals(signals: pd.DataFrame):
    in_pos = False
    filtered_entry = pd.Series(False, index=signals.index)
    filtered_exit = pd.Series(False, index=signals.index)

    for ts in signals.index:
        if not in_pos and signals['entry_rise'].loc[ts]:
            filtered_entry.loc[ts] = True
            in_pos = True
        elif in_pos and signals['exit_rise'].loc[ts]:
            filtered_exit.loc[ts] = True
            in_pos = False
    signals['entry_signal'] = filtered_entry
    signals['exit_signal'] = filtered_exit
    return signals


def calc_signals(df: pd.DataFrame, adx_length: int = 14, adx_threshold: int = 25) -> pd.DataFrame:
    df_signals = df.copy()
    if not isinstance(df.columns, pd.MultiIndex):
        print("Not a multiindex dataframe")
        return df_signals

    tickers = df.columns.levels[0].tolist()

    for ticker in tickers:
        df_single = df.xs(ticker, axis=1, level=0)

        exit_df = exit_signals(df_single,
                               atr_multiplier=3.0,
                               ce_high_length=22)

        entry_df = entry_signals(df_single,
                                 chandelier_exit=exit_df['ce_exit'],
                                 adx_length=adx_length,
                                 adx_threshold=adx_threshold)

        entry_exit_df = pd.concat([entry_df, exit_df], axis=1)
        entry_exit_df = filter_signals(entry_exit_df) # one entry -> one exit
        entry_exit_df = add_ticker_to_columns(ticker, entry_exit_df) # add ticker to columns, ex: (BTC, entry_signal)
        df_signals = pd.concat([df_signals, entry_exit_df], axis=1)
    return df_signals.sort_index(axis=1, level=0)


signals = calc_signals(df_multi)
print(signals.info())
print(signals['BTC'].tail())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 6024 entries, 2023-01-01 00:00:00+00:00 to 2025-09-30 20:00:00+00:00
Freq: 4h
Columns: 600 entries, ('ADA', 'adx') to ('ZEC', 'volume')
dtypes: Int64(24), bool(240), float32(96), float64(192), object(48)
memory usage: 16.0+ MB
None
                                 adx  adx_bull      adx_d      adx_dmn  \
2025-09-30 04:00:00+00:00  44.102223      True  43.488119  1588.735296   
2025-09-30 08:00:00+00:00  42.691486      True  43.369257  2355.355766   
2025-09-30 12:00:00+00:00  41.381516      True  42.741869  2187.116068   
2025-09-30 16:00:00+00:00  40.863434      True  41.777460  2030.893492   
2025-09-30 20:00:00+00:00  40.575425      True  40.978470  1885.829671   

                               adx_dmp  adx_signal base  ce_exit  \
2025-09-30 04:00:00+00:00  4169.611808        True  BTC    False   
2025-09-30 08:00:00+00:00  3871.782393        True  BTC    False   
2025-09-30 12:00:00+00:00  3595.226508        True  BTC    False  